# Paper assistant (MCP client)

This notebook talks to the **paper-assistant MCP server** (`mcp_server/server.py`).
The server can (1) answer questions about one downloaded PDF using **RAG**, (2) summarize it, and
(3) run the existing multi-agent pipeline (`main.py`). The notebook only sends requests and shows the results.

**Before you start:** `pip install -r requirements.txt`, keys in `.env`, PDFs in `papers/`.
Run the cells from the top. Cells use top-level `await`.

In [1]:
# Setup: project paths and MCP client imports
import os
import sys
from pathlib import Path

import nbformat
from IPython.display import Markdown, display
from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SERVER = PROJECT / "mcp_server" / "server.py"
TIMEOUT_MIN = 30        # pipeline steps can take minutes
NOTES_NB = PROJECT / "notebooks" / "paper_notes.ipynb"     # where save_note() writes
SERVER_LOG = open(PROJECT / "notebooks" / "server.log", "a", encoding="utf-8")   # server log lines go here
print("server:", SERVER, "| exists:", SERVER.exists())

server: C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\mcp_server\server.py | exists: True


In [2]:
# The helper every cell uses: start the server (stdio), call ONE tool, return its text
def server_params():
    return StdioServerParameters(command=sys.executable, args=[str(SERVER)], env={**os.environ})


async def call(tool, **args):
    # errlog must be a real file: Jupyter's sys.stderr has no file handle (breaks on Windows)
    async with Client(stdio_client(server_params(), errlog=SERVER_LOG)) as client:
        res = await client.call_tool(tool, args, read_timeout_seconds=TIMEOUT_MIN * 60.0)
    text = "".join(getattr(c, "text", "") for c in res.content)
    if res.is_error:                      # a failing tool answers with is_error, it does not raise
        raise RuntimeError(text)
    return text


def show(text):
    display(Markdown(text))


def save_note(markdown_text, path=NOTES_NB):
    # append a markdown cell to a notebook file (created if missing)
    path = Path(path)
    nb = nbformat.read(path, as_version=4) if path.exists() else nbformat.v4.new_notebook()
    nb.cells.append(nbformat.v4.new_markdown_cell(markdown_text))
    nbformat.write(nb, path)
    print("saved to", path)

## 1. Which tools does the server offer?

In [3]:
# Ask the server what it can do (MCP discovery)
async def list_tools():
    async with Client(stdio_client(server_params(), errlog=SERVER_LOG)) as client:
        return (await client.list_tools()).tools


for t in await list_tools():
    print(f"- {t.name}: {(t.description or '').splitlines()[0]}")

- list_papers: List the PDFs found in the allowed folders (papers/ by default) and whether each is indexed.
- index_paper: Index one PDF (file name, part of the name, or full path) so it can be searched. Safe to repeat.
- index_folder: Index every PDF in a folder (default: the first allowed folder, normally papers/). Folder must be allowed.
- search_papers: Semantic search over indexed papers. Give `paper` to search only one PDF. Returns top chunks with page.
- ask_paper: Answer a specific question about ONE downloaded paper: retrieves the best passages (RAG), then the LLM
- summarize_paper: Summarize ONE paper (problem, method, results, limitations) from retrieved passages, with page citations.
- get_paper_sheet: Read the detailed sheet the pipeline's extractor already made (out/extracted_v3). With no `paper`, list the
- run_step: Run one stage of the paper pipeline (main.py --step ...). step: extract|analyze|literature|write|review|fix|
- extract_sheets: Stage 1: read every PDF in pa

## 2. Your papers
`list_papers` shows the PDFs in `papers/`. `index_folder` reads them into the RAG index
(only needed once per file; unchanged files are skipped).

In [4]:
show(await call("list_papers"))

- 1706.03762v7.pdf (2.2 MB) [indexed]  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\1706.03762v7.pdf
- 2201.11903v6.pdf (0.9 MB) [indexed]  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\2201.11903v6.pdf
- 2210.03629v3.pdf (0.6 MB) [indexed]  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\2210.03629v3.pdf
- 2302.04761v1.pdf (0.7 MB) [indexed]  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\2302.04761v1.pdf
- 2304.03442v2.pdf (11.9 MB) [indexed]  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\2304.03442v2.pdf

In [5]:
print(await call("index_folder"))

[
 {
  "name": "1706.03762v7.pdf",
  "status": "already indexed",
  "chunks": 49
 },
 {
  "name": "2201.11903v6.pdf",
  "status": "already indexed",
  "chunks": 174
 },
 {
  "name": "2210.03629v3.pdf",
  "status": "already indexed",
  "chunks": 138
 },
 {
  "name": "2302.04761v1.pdf",
  "status": "already indexed",
  "chunks": 89
 },
 {
  "name": "2304.03442v2.pdf",
  "status": "already indexed",
  "chunks": 159
 }
]


## 3. Ask a specific question about ONE paper (RAG)
Write the PDF name (or part of it) and your question. The server finds the best passages, and the LLM answers
**only from them**, with page numbers.

In [ ]:
PAPER = "1706.03762"          # file name or part of it
QUESTION = "How many BLEU points does the big model get on English-to-German, and how long was it trained?"

answer = await call("ask_paper", pdf=PAPER, question=QUESTION, k=6)
show(answer)

In [ ]:
# Write the answer into a notebook as a markdown cell
save_note(f"### {QUESTION}\n\n{answer}")

## 4. Summarize one paper
Retrieves passages for *problem, method, results, limitations*, then writes a cited summary.

In [ ]:
summary = await call("summarize_paper", pdf=PAPER)
show(summary)

In [ ]:
save_note(summary)

## 5. Search across all indexed papers
No LLM involved: just the most relevant passages, with file and page.

In [ ]:
show(await call("search_papers", query="how an agent stores and retrieves memories", k=4))

## 6. Use what the pipeline already extracted
The extractor already built a detailed sheet per paper (`out/extracted_v3`).

In [ ]:
print(await call("get_paper_sheet"))                    # list the sheets
print(await call("get_paper_sheet", paper="react", fields="title,limitations"))

## 7. Run the multi-agent pipeline, stage by stage
Each stage calls `python main.py --step ...` inside the server. They use the LLM and take minutes.

In [ ]:
show(await call("extract_sheets"))

In [ ]:
TOPIC = "Memory and tool use for LLM agents"
show(await call("analyze_topic", topic=TOPIC))

In [ ]:
show(await call("write_draft", topic=TOPIC, venue="Conference"))

In [ ]:
show(await call("review_draft"))

## 8. Read the results

In [ ]:
print(await call("list_outputs"))

In [ ]:
draft = await call("read_output", name="paper_draft.md", max_chars=6000)
show(draft)

### Writing into *this* notebook
`save_note()` writes to `notebooks/paper_notes.ipynb`. To write into this notebook instead, set
`NOTES_NB = PROJECT / "notebooks" / "paper_assistant.ipynb"` and then use **File > Reload Notebook from Disk**
(Jupyter does not refresh a notebook that was changed on disk behind its back).

In [10]:
import chromadb, numpy as np
from chromadb.config import Settings

client = chromadb.PersistentClient(str(PROJECT / ".rag_store"), settings=Settings(anonymized_telemetry=False))
print(client.list_collections())            # should show papers_default

col = client.get_collection("papers_default")
d = col.get(include=["embeddings", "documents", "metadatas"], limit=10)

for doc, meta, v in zip(d["documents"], d["metadatas"], d["embeddings"]):
    print(f'{meta["name"]} p.{meta["page"]}  length={len(v)}  first 5 numbers={np.round(v[:5], 3)}')
    print("   ", doc[:80], "\n")

[Collection(name=papers_default)]
1706.03762v7.pdf p.1  length=384  first 5 numbers=[-0.119 -0.129  0.035 -0.014  0.04 ]
    Provided proper attribution is provided, Google hereby grants permission to repr 

1706.03762v7.pdf p.1  length=384  first 5 numbers=[-0.059 -0.114  0.034  0.005  0.   ]
    a new simple network architecture, the Transformer, based solely on attention me 

1706.03762v7.pdf p.1  length=384  first 5 numbers=[-0.105 -0.094  0.006 -0.066 -0.002]
    with large and limited training data. ∗Equal contribution. Listing order is rand 

1706.03762v7.pdf p.1  length=384  first 5 numbers=[-0.082 -0.085  0.074 -0.013  0.012]
    our earlier codebase, greatly improving results and massively accelerating our r 

1706.03762v7.pdf p.2  length=384  first 5 numbers=[-0.065 -0.061  0.064 -0.003  0.005]
    1 Introduction Recurrent neural networks, long short-term memory [13] and gated  

1706.03762v7.pdf p.2  length=384  first 5 numbers=[-0.047 -0.061 -0.011 -0.007  0.   ]
    lengt

In [9]:
from chromadb.utils import embedding_functions
col = client.get_collection("papers_default", embedding_function=embedding_functions.DefaultEmbeddingFunction())

res = col.query(query_texts=["How long was the model trained and on what hardware?"], n_results=3,
                include=["documents", "metadatas", "distances"])

for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f'{meta["name"]} p.{meta["page"]}  distance={dist:.3f}')
    print("   ", doc[:150], "\n")

1706.03762v7.pdf p.7  distance=0.613
    the training regime for our models. 5.1 Training Data and Batching We trained on the standard WMT 2014 English-German dataset consisting of about 4.5  

1706.03762v7.pdf p.7  distance=0.650
    about 0.4 seconds. We trained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the bottom line of table 3),  

2210.03629v3.pdf p.7  distance=0.668
    of episodes where the chosen product satisﬁes all requirements) on 500 test instructions. We formulate Act prompts with actions to search, choose prod 

